In [2]:
# Verification block
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Pandas Version Check:", pd.__version__)
print("All analytical components are successfully verified and online!")


Pandas Version Check: 3.1.0
All analytical components are successfully verified and online!


In [ ]:
import os
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Establish the clean path boundaries targeting your renamed file
RAW_DATA_PATH = "../data/raw/influencers_raw.csv"
PROCESSED_DATA_PATH = "../data/processed/clean_influencer_data.csv"

# Load the verified dataset file
df = pd.read_csv(RAW_DATA_PATH)

# Clean string numbers into floats (e.g., '1.2M' -> 1200000, '45.5K' -> 45500)
def parse_metrics(val):
    if isinstance(val, str):
        val = val.upper().strip()
        if 'M' in val: return float(val.replace('M', '')) * 1000000
        if 'K' in val: return float(val.replace('K', '')) * 1000
    return float(val)

# Fix column names to stand out cleanly
df.columns = df.columns.str.title().str.strip()

# Target columns specifically found inside the Syed Jaferk Kaggle file
followers_col = [c for c in df.columns if 'Follower' in c][0]
engagement_col = [c for c in df.columns if 'Engagement Avg' in c or 'Engagement' in c][0]

df['Followers_Clean'] = df[followers_col].apply(parse_metrics)
df['Engagement_Avg_Clean'] = df[engagement_col].apply(parse_metrics)

# 2. Feature Engineering: Compute Key Performance Indicators (KPIs)
df['Engagement_Rate'] = (df['Engagement_Avg_Clean'] / df['Followers_Clean']) * 100
df['Engagement_Rate'] = df['Engagement_Rate'].round(4)

# Create output directories if missing
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../images", exist_ok=True)

# Save processed clean data structure safely
df.to_csv(PROCESSED_DATA_PATH, index=False)

# 3. Exploratory Data Analysis & Visualizations
plt.style.use('default')

# Visual 1: Category Niche Demand Distribution
category_col = [c for c in df.columns if 'Category' in c or 'Niche' in c][0]
plt.figure(figsize=(10, 5))
sns.countplot(
    data=df, 
    y=category_col, 
    order=df[category_col].value_counts().iloc[:10].index, 
    hue=category_col, 
    palette='muted', 
    legend=False
)
plt.title('Top 10 Influencer Niche Categories')
plt.savefig('../images/category_distribution.png', bbox_inches='tight')
plt.close()

# Visual 2: Strategic Engagement Performance Scatterplot
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Followers_Clean', y='Engagement_Rate')
plt.xscale('log')
plt.title('Follower Scaling vs. True Engagement Rate %')
plt.xlabel('Followers (Log Scale)')
plt.savefig('../images/follower_vs_engagement.png', bbox_inches='tight')
plt.close()

print("Analytics pipeline ran successfully! Check your data/processed/ and images/ folders.")
